# Experimentos Univariados 1.1–1.19

**Tesis MEC** — Comparación TSFMs vs Modelos Clásicos bajo DGPs controlados  
**Setup:** T ∈ {200, 1000} | H = 24 | R_LIST = [500] | Semilla = 3649  
**Métricas punto:** Bias, Varianza, MSE, RMSE, MAE  
**Métricas probabilísticas:** CRPS, Cobertura 80%/95%, Amplitud 80%/95%, Winkler Score 80%/95%  
**Resultados:** guardados en `results/univariate/` — si existen se cargan sin re-simular

---

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

from mectesis.dgp import (
    AR1, RandomWalk, AR1WithTrend, SeasonalDGP, AR1WithBreak,
    AR1ARCH, AR1GARCH, PureGARCH, AR1GJRGARCH,
    LocalLevelDGP, LocalTrendDGP, DampedTrendDGP,
    DeterministicSeasonalDGP, SeasonalRandomWalkDGP, LocalLevelSeasonalDGP,
)
from mectesis.models import (
    ARIMAModel, ChronosModel,
    SARIMAModel, ARIMAWithTrendModel, ARIMAWithBreakModel,
    ARARCHModel, ARGARCHModel, GARCHModel, ARGJRGARCHModel,
    SeasonalNaiveModel, ETSModel, ThetaModel,
)
from mectesis.simulation import MonteCarloEngine

# ── Parámetros globales ───────────────────────────────────────────────────
SEED    = 3649
H       = 24
R_LIST  = [500]          # agregar 1000 para robustez: [500, 1000]
T_LIST  = [50, 200]
RESULTS = Path("results/univariate")
RESULTS.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({"figure.dpi": 110, "font.size": 10})
pd.set_option("display.float_format", "{:.4f}".format)

print("Cargando Chronos-2 (puede tardar ~30 s)...")
chronos = ChronosModel(device="cpu")
print("Chronos-2 listo.")

In [ ]:
# ─── Funciones auxiliares ───────────────────────────────────────────────────

def _cache_path(exp_id: str, T: int, R: int) -> Path:
    return RESULTS / f"exp_{exp_id.replace('.', '_')}_T{T}_R{R}.csv"


def _save_results(results: dict, path: Path):
    """Guarda {model_name: DataFrame} como CSV con columna 'model'."""
    frames = []
    for mname, df in results.items():
        tmp = df.copy()
        tmp.insert(0, 'model', mname)
        frames.append(tmp)
    pd.concat(frames, ignore_index=True).to_csv(path, index=False)


def _load_results(path: Path) -> dict:
    """Carga CSV de vuelta a {model_name: DataFrame}."""
    df = pd.read_csv(path)
    return {
        mname: grp.drop(columns='model').reset_index(drop=True)
        for mname, grp in df.groupby('model', sort=False)
    }


def run_exp(dgp, make_models_fn, dgp_params, exp_id,
            T_list=T_LIST, R_list=R_LIST, H=H, seed=SEED):
    """
    Corre MC para todas las combinaciones (T, R).
    Si el CSV ya existe, lo carga sin re-simular.
    Retorna {(T, R): {model_name: DataFrame}}.
    """
    n_runs = len(T_list) * len(R_list)
    combos = ', '.join(f'(T={t}, R={r})' for t in T_list for r in R_list)
    print(f'Exp {exp_id}: {n_runs} ejecución(es) programada(s) → {combos}')

    all_results = {}
    for T in T_list:
        for R in R_list:
            cache = _cache_path(exp_id, T, R)
            if cache.exists():
                print(f'  T={T}, R={R}: cargando {cache.name} ...')
                all_results[(T, R)] = _load_results(cache)
                continue

            print(f'  T={T}, R={R}: simulando ...', end=' ', flush=True)
            dgp.rng = np.random.default_rng(seed)
            models = make_models_fn(T)
            engine = MonteCarloEngine(dgp, models, seed=seed)
            t0 = time.time()
            results = engine.run_monte_carlo(
                R, T, H, dgp_params, verbose=False)
            elapsed = time.time() - t0
            print(f'OK ({elapsed:.0f}s)')

            _save_results(results, cache)
            all_results[(T, R)] = results

    return all_results   # {(T, R): {model_name: DataFrame}}


def compute_blocks(results_TR: dict):
    """Dado {model_name: df}, calcula promedios h=1-12 y h=13-24."""
    out = {}
    for mname, df in results_TR.items():
        df_h = df[df["horizon"] != "avg_all"].copy()
        df_h["horizon"] = pd.to_numeric(df_h["horizon"], errors="coerce")
        out[mname] = {
            "h=1-12":  df_h[df_h["horizon"] <= 12].mean(numeric_only=True),
            "h=13-24": df_h[df_h["horizon"] >= 13].mean(numeric_only=True),
        }
    return out


def results_table(all_results):
    """Muestra tabla comparativa de métricas por bloque para todos los (T, R)."""
    # Detect available metric columns from first DataFrame
    sample_df = next(iter(next(iter(all_results.values())).values()))
    numeric_cols = [c for c in sample_df.columns
                    if c not in ('horizon',) and sample_df[c].dtype != object]

    rows = []
    for (T, R), res_TR in sorted(all_results.items()):
        for mname, blk in compute_blocks(res_TR).items():
            for bname, m in blk.items():
                row = {'T': T, 'R': R, 'Modelo': mname, 'Bloque': bname}
                for col in numeric_cols:
                    if col in m.index:
                        row[col] = round(float(m[col]), 4)
                rows.append(row)

    df = pd.DataFrame(rows).set_index(['T', 'R', 'Modelo', 'Bloque'])
    grad_cols = [c for c in ['rmse', 'mae'] if c in df.columns]
    display(df.style.format(precision=4)
              .background_gradient(subset=grad_cols, cmap='YlOrRd'))


def plot_rep(dgp, make_models_fn, dgp_params,
             T=200, H=H, seed=SEED, title=''):
    """Visualización de una simulación representativa."""
    dgp_r = dgp.__class__(seed=seed)
    y = dgp_r.simulate(T=T, **dgp_params)
    y_train, y_test = y[:-H], y[-H:]
    models = make_models_fn(T)

    fig, ax = plt.subplots(figsize=(13, 4))
    x_tr = np.arange(len(y_train))
    x_te = np.arange(len(y_train), T)

    ax.plot(x_tr, y_train, color="steelblue", lw=1.4, alpha=0.85, label="Histórico")
    ax.plot(x_te, y_test, "k--", lw=1.5, label="Observado (test)")
    ax.axvline(len(y_train) - 0.5, color='gray', ls=':', lw=1, alpha=0.6)

    palette = ['crimson', 'darkorange', 'seagreen', 'purple', 'teal', 'olive']
    for i, m in enumerate(models):
        m.fit(y_train)
        y_hat = m.forecast(H)
        ax.plot(x_te, y_hat, color=palette[i % len(palette)],
                lw=1.5, marker='o', ms=3, label=m.name)
        if m.supports_intervals:
            lo, hi = m.forecast_intervals(H, level=0.95)
            ax.fill_between(x_te, lo, hi,
                            color=palette[i % len(palette)],
                            alpha=0.12, label='_nolegend_')

    ax.set(title=title, xlabel='t', ylabel='$Y_t$')
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()


def plot_metrics(all_results, title='', metrics=('rmse', 'bias')):
    """Gráficos de métricas vs h=1..24 por modelo, subplots por (T, R)."""
    keys = sorted(all_results.keys())
    fig, axes = plt.subplots(
        len(metrics), len(keys),
        figsize=(7 * len(keys), 3.5 * len(metrics)),
        squeeze=False,
    )
    palette = ['crimson', 'darkorange', 'seagreen', 'purple', 'teal', 'steelblue']

    for col, (T, R) in enumerate(keys):
        for row, metric in enumerate(metrics):
            ax = axes[row][col]
            for i, (mname, df) in enumerate(all_results[(T, R)].items()):
                df_h = df[df["horizon"] != "avg_all"].copy()
                df_h["horizon"] = pd.to_numeric(df_h["horizon"], errors="coerce")
                if metric not in df_h.columns:
                    continue
                ax.plot(df_h['horizon'], df_h[metric],
                        label=mname, color=palette[i % len(palette)], lw=1.5)
            ax.axvline(12.5, color='gray', ls=':', lw=0.8, alpha=0.5)
            ax.set(
                title=f'T={T}, R={R} — {metric.upper()}',
                xlabel='Horizonte h',
                ylabel=metric.upper(),
            )
            ax.legend(fontsize=8)

    fig.suptitle(title, fontsize=12)
    plt.tight_layout()
    plt.show()

---
## Experimento 1.1

**DGP:** AR(1) baja persistencia — $Y_t = 0.3\,Y_{t-1} + \varepsilon_t$  
**Core:** ARIMA(1,0,0), Chronos-2  
**Adicionales (no implementados aquí):** ETS(A,N,N), Theta, Naive, Drift

In [ ]:
dgp_1_1         = AR1(seed=SEED)
make_models_1_1 = lambda T: [ARIMAModel((1,0,0)), chronos]
dgp_params_1_1  = {'phi': 0.3}

results_1_1 = run_exp(
    dgp_1_1, make_models_1_1, dgp_params_1_1,
    exp_id='1.1',
)

In [ ]:
# Visualización representativa con banda de intervalo 95% (T=T_LIST[0])
plot_rep(
    dgp_1_1, make_models_1_1, dgp_params_1_1,
    T=T_LIST[0], title=f"Exp 1.1: AR(1) baja persistencia — Simulación representativa (T={T_LIST[0]}, seed={SEED})"
)

# Visualización representativa con banda de intervalo 95% (T=T_LIST[1])
plot_rep(
    dgp_1_1, make_models_1_1, dgp_params_1_1,
    T=T_LIST[1], title=f"Exp 1.1: AR(1) baja persistencia — Simulación representativa (T={T_LIST[1]}, seed={SEED})"
)

# Tabla de métricas por bloque
print("Tabla de métricas — Exp 1.1")
results_table(results_1_1)

# Gráficos de métricas por horizonte
plot_metrics(
    results_1_1,
    title=f"Exp 1.1 — Métricas por horizonte",
    metrics=("rmse", "crps", "winkler_95", "bias")
)

---
## Experimento 1.2

**DGP:** AR(1) alta persistencia — $Y_t = 0.9\,Y_{t-1} + \varepsilon_t$  
**Core:** ARIMA(1,0,0), Chronos-2  
**Adicionales:** ETS(A,A,N), Theta, Naive

In [ ]:
dgp_1_2         = AR1(seed=SEED)
make_models_1_2 = lambda T: [ARIMAModel((1,0,0)), chronos]
dgp_params_1_2  = {'phi': 0.9}

results_1_2 = run_exp(
    dgp_1_2, make_models_1_2, dgp_params_1_2,
    exp_id='1.2',
)

In [ ]:
# Visualización representativa con banda de intervalo 95% (T=T_LIST[0])
plot_rep(
    dgp_1_2, make_models_1_2, dgp_params_1_2,
    T=T_LIST[0], title=f"Exp 1.2: AR(1) alta persistencia — Simulación representativa (T={T_LIST[0]}, seed={SEED})"
)

# Visualización representativa con banda de intervalo 95% (T=T_LIST[1])
plot_rep(
    dgp_1_2, make_models_1_2, dgp_params_1_2,
    T=T_LIST[1], title=f"Exp 1.2: AR(1) alta persistencia — Simulación representativa (T={T_LIST[1]}, seed={SEED})"
)

# Tabla de métricas por bloque
print("Tabla de métricas — Exp 1.2")
results_table(results_1_2)

# Gráficos de métricas por horizonte
plot_metrics(
    results_1_2,
    title=f"Exp 1.2 — Métricas por horizonte",
    metrics=("rmse", "crps", "winkler_95", "bias")
)

---
## Experimento 1.3

**DGP:** Random Walk I(1) sin drift — $Y_t = Y_{t-1} + \varepsilon_t$  
**Core:** ARIMA(0,1,0), Chronos-2  
**Adicionales:** ETS(A,A,N), Theta, Drift

In [ ]:
dgp_1_3         = RandomWalk(seed=SEED)
make_models_1_3 = lambda T: [ARIMAModel((0,1,0)), chronos]
dgp_params_1_3  = {'drift': 0.0}

results_1_3 = run_exp(
    dgp_1_3, make_models_1_3, dgp_params_1_3,
    exp_id='1.3',
)

In [ ]:
# Visualización representativa con banda de intervalo 95% (T=T_LIST[0])
plot_rep(
    dgp_1_3, make_models_1_3, dgp_params_1_3,
    T=T_LIST[0], title=f"Exp 1.3: Random Walk sin drift — Simulación representativa (T={T_LIST[0]}, seed={SEED})"
)

# Visualización representativa con banda de intervalo 95% (T=T_LIST[1])
plot_rep(
    dgp_1_3, make_models_1_3, dgp_params_1_3,
    T=T_LIST[1], title=f"Exp 1.3: Random Walk sin drift — Simulación representativa (T={T_LIST[1]}, seed={SEED})"
)

# Tabla de métricas por bloque
print("Tabla de métricas — Exp 1.3")
results_table(results_1_3)

# Gráficos de métricas por horizonte
plot_metrics(
    results_1_3,
    title=f"Exp 1.3 — Métricas por horizonte",
    metrics=("rmse", "crps", "winkler_95", "bias")
)

---
## Experimento 1.4

**DGP:** Random Walk I(1) con drift — $Y_t = 0.5 + Y_{t-1} + \varepsilon_t$  
**Core:** ARIMA(0,1,0), Chronos-2  
**Adicionales:** ETS(A,A,N), Theta, Drift

In [ ]:
dgp_1_4         = RandomWalk(seed=SEED)
make_models_1_4 = lambda T: [ARIMAModel((0,1,0)), chronos]
dgp_params_1_4  = {'drift': 0.5}

results_1_4 = run_exp(
    dgp_1_4, make_models_1_4, dgp_params_1_4,
    exp_id='1.4',
)

In [ ]:
# Visualización representativa con banda de intervalo 95% (T=T_LIST[0])
plot_rep(
    dgp_1_4, make_models_1_4, dgp_params_1_4,
    T=T_LIST[0], title=f"Exp 1.4: Random Walk con drift — Simulación representativa (T={T_LIST[0]}, seed={SEED})"
)

# Visualización representativa con banda de intervalo 95% (T=T_LIST[1])
plot_rep(
    dgp_1_4, make_models_1_4, dgp_params_1_4,
    T=T_LIST[1], title=f"Exp 1.4: Random Walk con drift — Simulación representativa (T={T_LIST[1]}, seed={SEED})"
)

# Tabla de métricas por bloque
print("Tabla de métricas — Exp 1.4")
results_table(results_1_4)

# Gráficos de métricas por horizonte
plot_metrics(
    results_1_4,
    title=f"Exp 1.4 — Métricas por horizonte",
    metrics=("rmse", "crps", "winkler_95", "bias")
)

---
## Experimento 1.5

**DGP:** AR(1) + tendencia — $Y_t = 5 + 0.1t + 0.6\,Y_{t-1} + \varepsilon_t$  
**Core:** ARIMA(1,0,0)+trend (trend='ct'), Chronos-2  
**Adicionales:** Holt-Winters, ETS, Theta

In [ ]:
dgp_1_5         = AR1WithTrend(seed=SEED)
make_models_1_5 = lambda T: [ARIMAWithTrendModel((1,0,0), trend='ct'), chronos]
dgp_params_1_5  = {'intercept': 5.0, 'trend_coeff': 0.1, 'phi': 0.6}

results_1_5 = run_exp(
    dgp_1_5, make_models_1_5, dgp_params_1_5,
    exp_id='1.5',
)

In [ ]:
# Visualización representativa con banda de intervalo 95% (T=T_LIST[0])
plot_rep(
    dgp_1_5, make_models_1_5, dgp_params_1_5,
    T=T_LIST[0], title=f"Exp 1.5: AR(1) con tendencia — Simulación representativa (T={T_LIST[0]}, seed={SEED})"
)

# Visualización representativa con banda de intervalo 95% (T=T_LIST[1])
plot_rep(
    dgp_1_5, make_models_1_5, dgp_params_1_5,
    T=T_LIST[1], title=f"Exp 1.5: AR(1) con tendencia — Simulación representativa (T={T_LIST[1]}, seed={SEED})"
)

# Tabla de métricas por bloque
print("Tabla de métricas — Exp 1.5")
results_table(results_1_5)

# Gráficos de métricas por horizonte
plot_metrics(
    results_1_5,
    title=f"Exp 1.5 — Métricas por horizonte",
    metrics=("rmse", "crps", "winkler_95", "bias")
)

---
## Experimento 1.6

**DGP:** SARIMA trimestral (s=4) — $(1-0.5L)(1-0.3L^4)Y_t = \varepsilon_t$  
**Core:** SARIMA(1,0,0)(1,0,0)_4, Chronos-2  
**Adicionales:** ETS(A,A,A), Seasonal Naive

In [ ]:
dgp_1_6         = SeasonalDGP(seed=SEED)
make_models_1_6 = lambda T: [SARIMAModel((1,0,0),(1,0,0,4)), chronos]
dgp_params_1_6  = {'phi': 0.5, 'Phi': 0.3, 's': 4, 'integrated': False}

results_1_6 = run_exp(
    dgp_1_6, make_models_1_6, dgp_params_1_6,
    exp_id='1.6',
)

In [ ]:
# Visualización representativa con banda de intervalo 95% (T=T_LIST[0])
plot_rep(
    dgp_1_6, make_models_1_6, dgp_params_1_6,
    T=T_LIST[0], title=f"Exp 1.6: SARIMA trimestral (s=4) — Simulación representativa (T={T_LIST[0]}, seed={SEED})"
)

# Visualización representativa con banda de intervalo 95% (T=T_LIST[1])
plot_rep(
    dgp_1_6, make_models_1_6, dgp_params_1_6,
    T=T_LIST[1], title=f"Exp 1.6: SARIMA trimestral (s=4) — Simulación representativa (T={T_LIST[1]}, seed={SEED})"
)

# Tabla de métricas por bloque
print("Tabla de métricas — Exp 1.6")
results_table(results_1_6)

# Gráficos de métricas por horizonte
plot_metrics(
    results_1_6,
    title=f"Exp 1.6 — Métricas por horizonte",
    metrics=("rmse", "crps", "winkler_95", "bias")
)

---
## Experimento 1.7

**DGP:** SARIMA mensual (s=12) — $(1-L)(1-L^{12})Y_t = \varepsilon_t$  
**Core:** SARIMA(0,1,0)(0,1,0)_12, Chronos-2  
**Adicionales:** Holt-Winters multiplicativo, ETS, Seasonal Naive

In [ ]:
dgp_1_7         = SeasonalDGP(seed=SEED)
make_models_1_7 = lambda T: [SARIMAModel((0,1,0),(0,1,0,12)), chronos]
dgp_params_1_7  = {'integrated': True, 's': 12}

results_1_7 = run_exp(
    dgp_1_7, make_models_1_7, dgp_params_1_7,
    exp_id='1.7',
)

In [ ]:
# Visualización representativa con banda de intervalo 95% (T=T_LIST[0])
plot_rep(
    dgp_1_7, make_models_1_7, dgp_params_1_7,
    T=T_LIST[0], title=f"Exp 1.7: SARIMA mensual (s=12) — Simulación representativa (T={T_LIST[0]}, seed={SEED})"
)

# Visualización representativa con banda de intervalo 95% (T=T_LIST[1])
plot_rep(
    dgp_1_7, make_models_1_7, dgp_params_1_7,
    T=T_LIST[1], title=f"Exp 1.7: SARIMA mensual (s=12) — Simulación representativa (T={T_LIST[1]}, seed={SEED})"
)

# Tabla de métricas por bloque
print("Tabla de métricas — Exp 1.7")
results_table(results_1_7)

# Gráficos de métricas por horizonte
plot_metrics(
    results_1_7,
    title=f"Exp 1.7 — Métricas por horizonte",
    metrics=("rmse", "crps", "winkler_95", "bias")
)

---
## Experimento 1.8

**DGP:** AR(1) con quiebre en $T/2$ — $\phi$ cambia de 0.3 a 0.8  
**Core:** ARIMA(1,0,0)+break (dummy exógena), Chronos-2  
**Adicionales:** ARIMA(1,0,0) sin quiebre, ETS

In [ ]:
dgp_1_8         = AR1WithBreak(seed=SEED)
make_models_1_8 = lambda T: [ARIMAWithBreakModel((1,0,0), T_total=T), chronos]
dgp_params_1_8  = {'phi_before': 0.3, 'phi_after': 0.8}

results_1_8 = run_exp(
    dgp_1_8, make_models_1_8, dgp_params_1_8,
    exp_id='1.8',
)

In [ ]:
# Visualización representativa con banda de intervalo 95% (T=T_LIST[0])
plot_rep(
    dgp_1_8, make_models_1_8, dgp_params_1_8,
    T=T_LIST[0], title=f"Exp 1.8: AR(1) con quiebre estructural — Simulación representativa (T={T_LIST[0]}, seed={SEED})"
)

# Visualización representativa con banda de intervalo 95% (T=T_LIST[1])
plot_rep(
    dgp_1_8, make_models_1_8, dgp_params_1_8,
    T=T_LIST[1], title=f"Exp 1.8: AR(1) con quiebre estructural — Simulación representativa (T={T_LIST[1]}, seed={SEED})"
)

# Tabla de métricas por bloque
print("Tabla de métricas — Exp 1.8")
results_table(results_1_8)

# Gráficos de métricas por horizonte
plot_metrics(
    results_1_8,
    title=f"Exp 1.8 — Métricas por horizonte",
    metrics=("rmse", "crps", "winkler_95", "bias")
)

---
## Experimento 1.9

**DGP:** AR(1)–ARCH(1) — $Y_t = 0.3Y_{t-1} + \varepsilon_t$; $\sigma_t^2 = 0.1 + 0.3\,\varepsilon_{t-1}^2$  
**Core:** AR(1)+ARCH(1), Chronos-2

In [ ]:
dgp_1_9         = AR1ARCH(seed=SEED)
make_models_1_9 = lambda T: [ARARCHModel(), chronos]
dgp_params_1_9  = {'phi': 0.3, 'omega': 0.1, 'alpha': 0.3}

results_1_9 = run_exp(
    dgp_1_9, make_models_1_9, dgp_params_1_9,
    exp_id='1.9',
)

In [ ]:
# Visualización representativa con banda de intervalo 95% (T=T_LIST[0])
plot_rep(
    dgp_1_9, make_models_1_9, dgp_params_1_9,
    T=T_LIST[0], title=f"Exp 1.9: AR(1)–ARCH(1) — Simulación representativa (T={T_LIST[0]}, seed={SEED})"
)

# Visualización representativa con banda de intervalo 95% (T=T_LIST[1])
plot_rep(
    dgp_1_9, make_models_1_9, dgp_params_1_9,
    T=T_LIST[1], title=f"Exp 1.9: AR(1)–ARCH(1) — Simulación representativa (T={T_LIST[1]}, seed={SEED})"
)

# Tabla de métricas por bloque
print("Tabla de métricas — Exp 1.9")
results_table(results_1_9)

# Gráficos de métricas por horizonte
plot_metrics(
    results_1_9,
    title=f"Exp 1.9 — Métricas por horizonte",
    metrics=("rmse", "crps", "winkler_95", "bias")
)

---
## Experimento 1.10

**DGP:** AR(1)–GARCH(1,1) — $Y_t = 0.3Y_{t-1} + \varepsilon_t$; $\sigma_t^2 = 0.1 + 0.1\,\varepsilon_{t-1}^2 + 0.8\,\sigma_{t-1}^2$  
**Core:** AR(1)+GARCH(1,1), Chronos-2

In [ ]:
dgp_1_10         = AR1GARCH(seed=SEED)
make_models_1_10 = lambda T: [ARGARCHModel(), chronos]
dgp_params_1_10  = {'phi': 0.3, 'omega': 0.1, 'alpha': 0.1, 'beta': 0.8}

results_1_10 = run_exp(
    dgp_1_10, make_models_1_10, dgp_params_1_10,
    exp_id='1.10',
)

In [ ]:
# Visualización representativa con banda de intervalo 95% (T=T_LIST[0])
plot_rep(
    dgp_1_10, make_models_1_10, dgp_params_1_10,
    T=T_LIST[0], title=f"Exp 1.10: AR(1)–GARCH(1,1) — Simulación representativa (T={T_LIST[0]}, seed={SEED})"
)

# Visualización representativa con banda de intervalo 95% (T=T_LIST[1])
plot_rep(
    dgp_1_10, make_models_1_10, dgp_params_1_10,
    T=T_LIST[1], title=f"Exp 1.10: AR(1)–GARCH(1,1) — Simulación representativa (T={T_LIST[1]}, seed={SEED})"
)

# Tabla de métricas por bloque
print("Tabla de métricas — Exp 1.10")
results_table(results_1_10)

# Gráficos de métricas por horizonte
plot_metrics(
    results_1_10,
    title=f"Exp 1.10 — Métricas por horizonte",
    metrics=("rmse", "crps", "winkler_95", "bias")
)

---
## Experimento 1.11

**DGP:** GARCH(1,1) media cero — $Y_t = \sigma_t z_t$; $\sigma_t^2 = 0.1 + 0.1\,Y_{t-1}^2 + 0.8\,\sigma_{t-1}^2$  
**Core:** GARCH(1,1) media cero, Chronos-2

In [ ]:
dgp_1_11         = PureGARCH(seed=SEED)
make_models_1_11 = lambda T: [GARCHModel(), chronos]
dgp_params_1_11  = {'omega': 0.1, 'alpha': 0.1, 'beta': 0.8}

results_1_11 = run_exp(
    dgp_1_11, make_models_1_11, dgp_params_1_11,
    exp_id='1.11',
)

In [ ]:
# Visualización representativa con banda de intervalo 95% (T=T_LIST[0])
plot_rep(
    dgp_1_11, make_models_1_11, dgp_params_1_11,
    T=T_LIST[0], title=f"Exp 1.11: GARCH(1,1) media cero — Simulación representativa (T={T_LIST[0]}, seed={SEED})"
)

# Visualización representativa con banda de intervalo 95% (T=T_LIST[1])
plot_rep(
    dgp_1_11, make_models_1_11, dgp_params_1_11,
    T=T_LIST[1], title=f"Exp 1.11: GARCH(1,1) media cero — Simulación representativa (T={T_LIST[1]}, seed={SEED})"
)

# Tabla de métricas por bloque
print("Tabla de métricas — Exp 1.11")
results_table(results_1_11)

# Gráficos de métricas por horizonte
plot_metrics(
    results_1_11,
    title=f"Exp 1.11 — Métricas por horizonte",
    metrics=("rmse", "crps", "winkler_95", "bias")
)

---
## Experimento 1.12

**DGP:** AR(1)–GJR–GARCH — $\sigma_t^2 = 0.1 + 0.05\,\varepsilon_{t-1}^2 + 0.1\,\varepsilon_{t-1}^2\mathbf{1}\{\varepsilon_{t-1}<0\} + 0.8\,\sigma_{t-1}^2$  
**Core:** AR(1)+GJR-GARCH(1,1,1), Chronos-2

In [ ]:
dgp_1_12         = AR1GJRGARCH(seed=SEED)
make_models_1_12 = lambda T: [ARGJRGARCHModel(), chronos]
dgp_params_1_12  = {'phi': 0.3, 'omega': 0.1, 'alpha': 0.05, 'gamma': 0.1, 'beta': 0.8}

results_1_12 = run_exp(
    dgp_1_12, make_models_1_12, dgp_params_1_12,
    exp_id='1.12',
)

In [ ]:
# Visualización representativa con banda de intervalo 95% (T=T_LIST[0])
plot_rep(
    dgp_1_12, make_models_1_12, dgp_params_1_12,
    T=T_LIST[0], title=f"Exp 1.12: AR(1)–GJR–GARCH — Simulación representativa (T={T_LIST[0]}, seed={SEED})"
)

# Visualización representativa con banda de intervalo 95% (T=T_LIST[1])
plot_rep(
    dgp_1_12, make_models_1_12, dgp_params_1_12,
    T=T_LIST[1], title=f"Exp 1.12: AR(1)–GJR–GARCH — Simulación representativa (T={T_LIST[1]}, seed={SEED})"
)

# Tabla de métricas por bloque
print("Tabla de métricas — Exp 1.12")
results_table(results_1_12)

# Gráficos de métricas por horizonte
plot_metrics(
    results_1_12,
    title=f"Exp 1.12 — Métricas por horizonte",
    metrics=("rmse", "crps", "winkler_95", "bias")
)

---
## Experimento 1.13

**DGP:** Nivel local — $\ell_t = \ell_{t-1} + \eta_t$, $Y_t = \ell_t + \varepsilon_t$  
**Core:** ETS(A,N,N), Chronos-2

In [ ]:
dgp_1_13         = LocalLevelDGP(seed=SEED)
make_models_1_13 = lambda T: [ETSModel(trend=None), chronos]
dgp_params_1_13  = {}

results_1_13 = run_exp(
    dgp_1_13, make_models_1_13, dgp_params_1_13,
    exp_id='1.13',
)

In [ ]:
# Visualización representativa con banda de intervalo 95% (T=T_LIST[0])
plot_rep(
    dgp_1_13, make_models_1_13, dgp_params_1_13,
    T=T_LIST[0], title=f"Exp 1.13: Local level — ETS(A,N,N) — Simulación representativa (T={T_LIST[0]}, seed={SEED})"
)

# Visualización representativa con banda de intervalo 95% (T=T_LIST[1])
plot_rep(
    dgp_1_13, make_models_1_13, dgp_params_1_13,
    T=T_LIST[1], title=f"Exp 1.13: Local level — ETS(A,N,N) — Simulación representativa (T={T_LIST[1]}, seed={SEED})"
)

# Tabla de métricas por bloque
print("Tabla de métricas — Exp 1.13")
results_table(results_1_13)

# Gráficos de métricas por horizonte
plot_metrics(
    results_1_13,
    title=f"Exp 1.13 — Métricas por horizonte",
    metrics=("rmse", "crps", "winkler_95", "bias")
)

---
## Experimento 1.14

**DGP:** Tendencia local (LLT) — $\ell_t = \ell_{t-1} + b_{t-1} + \eta_t$, $b_t = b_{t-1} + \zeta_t$  
**Core:** ETS(A,A,N), Chronos-2

In [ ]:
dgp_1_14         = LocalTrendDGP(seed=SEED)
make_models_1_14 = lambda T: [ETSModel(trend='add'), chronos]
dgp_params_1_14  = {}

results_1_14 = run_exp(
    dgp_1_14, make_models_1_14, dgp_params_1_14,
    exp_id='1.14',
)

In [ ]:
# Visualización representativa con banda de intervalo 95% (T=T_LIST[0])
plot_rep(
    dgp_1_14, make_models_1_14, dgp_params_1_14,
    T=T_LIST[0], title=f"Exp 1.14: Local trend — ETS(A,A,N) — Simulación representativa (T={T_LIST[0]}, seed={SEED})"
)

# Visualización representativa con banda de intervalo 95% (T=T_LIST[1])
plot_rep(
    dgp_1_14, make_models_1_14, dgp_params_1_14,
    T=T_LIST[1], title=f"Exp 1.14: Local trend — ETS(A,A,N) — Simulación representativa (T={T_LIST[1]}, seed={SEED})"
)

# Tabla de métricas por bloque
print("Tabla de métricas — Exp 1.14")
results_table(results_1_14)

# Gráficos de métricas por horizonte
plot_metrics(
    results_1_14,
    title=f"Exp 1.14 — Métricas por horizonte",
    metrics=("rmse", "crps", "winkler_95", "bias")
)

---
## Experimento 1.15

**DGP:** Tendencia amortiguada — $\ell_t = \ell_{t-1} + \phi b_{t-1} + \eta_t$, $b_t = \phi b_{t-1} + \zeta_t$ ($\phi=0.9$)  
**Core:** ETS(A,Ad,N), Chronos-2

In [ ]:
dgp_1_15         = DampedTrendDGP(seed=SEED)
make_models_1_15 = lambda T: [ETSModel(trend='add', damped_trend=True), chronos]
dgp_params_1_15  = {'phi': 0.9}

results_1_15 = run_exp(
    dgp_1_15, make_models_1_15, dgp_params_1_15,
    exp_id='1.15',
)

In [ ]:
# Visualización representativa con banda de intervalo 95% (T=T_LIST[0])
plot_rep(
    dgp_1_15, make_models_1_15, dgp_params_1_15,
    T=T_LIST[0], title=f"Exp 1.15: Damped trend — ETS(A,Ad,N) — Simulación representativa (T={T_LIST[0]}, seed={SEED})"
)

# Visualización representativa con banda de intervalo 95% (T=T_LIST[1])
plot_rep(
    dgp_1_15, make_models_1_15, dgp_params_1_15,
    T=T_LIST[1], title=f"Exp 1.15: Damped trend — ETS(A,Ad,N) — Simulación representativa (T={T_LIST[1]}, seed={SEED})"
)

# Tabla de métricas por bloque
print("Tabla de métricas — Exp 1.15")
results_table(results_1_15)

# Gráficos de métricas por horizonte
plot_metrics(
    results_1_15,
    title=f"Exp 1.15 — Métricas por horizonte",
    metrics=("rmse", "crps", "winkler_95", "bias")
)

---
## Experimento 1.16

**DGP:** Estacionalidad determinística pura (s=12) — $Y_t = \mu + s_t + \varepsilon_t$, $s_t = s_{t-12}$  
**Core:** Seasonal Naive (s=12), Chronos-2

In [ ]:
dgp_1_16         = DeterministicSeasonalDGP(seed=SEED)
make_models_1_16 = lambda T: [SeasonalNaiveModel(period=12), chronos]
dgp_params_1_16  = {'mu': 5.0, 'sigma_eps': 1.0, 's': 12}

results_1_16 = run_exp(
    dgp_1_16, make_models_1_16, dgp_params_1_16,
    exp_id='1.16',
)

In [ ]:
# Visualización representativa con banda de intervalo 95% (T=T_LIST[0])
plot_rep(
    dgp_1_16, make_models_1_16, dgp_params_1_16,
    T=T_LIST[0], title=f"Exp 1.16: Estacionalidad determinística (s=12) — Simulación representativa (T={T_LIST[0]}, seed={SEED})"
)

# Visualización representativa con banda de intervalo 95% (T=T_LIST[1])
plot_rep(
    dgp_1_16, make_models_1_16, dgp_params_1_16,
    T=T_LIST[1], title=f"Exp 1.16: Estacionalidad determinística (s=12) — Simulación representativa (T={T_LIST[1]}, seed={SEED})"
)

# Tabla de métricas por bloque
print("Tabla de métricas — Exp 1.16")
results_table(results_1_16)

# Gráficos de métricas por horizonte
plot_metrics(
    results_1_16,
    title=f"Exp 1.16 — Métricas por horizonte",
    metrics=("rmse", "crps", "winkler_95", "bias")
)

---
## Experimento 1.17

**DGP:** Seasonal random walk (s=12) — $Y_t = Y_{t-12} + \varepsilon_t$  
**Core:** Seasonal Naive (s=12), Chronos-2

In [ ]:
dgp_1_17         = SeasonalRandomWalkDGP(seed=SEED)
make_models_1_17 = lambda T: [SeasonalNaiveModel(period=12), chronos]
dgp_params_1_17  = {'s': 12, 'sigma': 1.0}

results_1_17 = run_exp(
    dgp_1_17, make_models_1_17, dgp_params_1_17,
    exp_id='1.17',
)

In [ ]:
# Visualización representativa con banda de intervalo 95% (T=T_LIST[0])
plot_rep(
    dgp_1_17, make_models_1_17, dgp_params_1_17,
    T=T_LIST[0], title=f"Exp 1.17: Seasonal random walk (s=12) — Simulación representativa (T={T_LIST[0]}, seed={SEED})"
)

# Visualización representativa con banda de intervalo 95% (T=T_LIST[1])
plot_rep(
    dgp_1_17, make_models_1_17, dgp_params_1_17,
    T=T_LIST[1], title=f"Exp 1.17: Seasonal random walk (s=12) — Simulación representativa (T={T_LIST[1]}, seed={SEED})"
)

# Tabla de métricas por bloque
print("Tabla de métricas — Exp 1.17")
results_table(results_1_17)

# Gráficos de métricas por horizonte
plot_metrics(
    results_1_17,
    title=f"Exp 1.17 — Métricas por horizonte",
    metrics=("rmse", "crps", "winkler_95", "bias")
)

---
## Experimento 1.18

**DGP:** Trend + seasonality (ETS A,A,A) — $Y_t = \ell_t + b_t + \gamma_t + \varepsilon_t$, s=12  
**Core:** ETS(A,A,A), Chronos-2

In [ ]:
dgp_1_18         = LocalLevelSeasonalDGP(seed=SEED)
make_models_1_18 = lambda T: [ETSModel(trend='add', seasonal='add', seasonal_periods=12), chronos]
dgp_params_1_18  = {}

results_1_18 = run_exp(
    dgp_1_18, make_models_1_18, dgp_params_1_18,
    exp_id='1.18',
)

In [ ]:
# Visualización representativa con banda de intervalo 95% (T=T_LIST[0])
plot_rep(
    dgp_1_18, make_models_1_18, dgp_params_1_18,
    T=T_LIST[0], title=f"Exp 1.18: Full ETS(A,A,A) — tendencia + estacionalidad — Simulación representativa (T={T_LIST[0]}, seed={SEED})"
)

# Visualización representativa con banda de intervalo 95% (T=T_LIST[1])
plot_rep(
    dgp_1_18, make_models_1_18, dgp_params_1_18,
    T=T_LIST[1], title=f"Exp 1.18: Full ETS(A,A,A) — tendencia + estacionalidad — Simulación representativa (T={T_LIST[1]}, seed={SEED})"
)

# Tabla de métricas por bloque
print("Tabla de métricas — Exp 1.18")
results_table(results_1_18)

# Gráficos de métricas por horizonte
plot_metrics(
    results_1_18,
    title=f"Exp 1.18 — Métricas por horizonte",
    metrics=("rmse", "crps", "winkler_95", "bias")
)

---
## Experimento 1.19

**DGP:** Tendencia lineal pura — $Y_t = 0.1t + \varepsilon_t$  
**Core:** Theta, Chronos-2

In [ ]:
dgp_1_19         = AR1WithTrend(seed=SEED)
make_models_1_19 = lambda T: [ThetaModel(), chronos]
dgp_params_1_19  = {'intercept': 0.0, 'trend_coeff': 0.1, 'phi': 0.0}

results_1_19 = run_exp(
    dgp_1_19, make_models_1_19, dgp_params_1_19,
    exp_id='1.19',
)

In [ ]:
# Visualización representativa con banda de intervalo 95% (T=T_LIST[0])
plot_rep(
    dgp_1_19, make_models_1_19, dgp_params_1_19,
    T=T_LIST[0], title=f"Exp 1.19: Tendencia lineal — Theta — Simulación representativa (T={T_LIST[0]}, seed={SEED})"
)

# Visualización representativa con banda de intervalo 95% (T=T_LIST[1])
plot_rep(
    dgp_1_19, make_models_1_19, dgp_params_1_19,
    T=T_LIST[1], title=f"Exp 1.19: Tendencia lineal — Theta — Simulación representativa (T={T_LIST[1]}, seed={SEED})"
)

# Tabla de métricas por bloque
print("Tabla de métricas — Exp 1.19")
results_table(results_1_19)

# Gráficos de métricas por horizonte
plot_metrics(
    results_1_19,
    title=f"Exp 1.19 — Métricas por horizonte",
    metrics=("rmse", "crps", "winkler_95", "bias")
)